In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Project root: parent of the notebooks folder
PROJECT_DIR = Path.cwd().parent

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DIR)
print("Processed data directory:", PROCESSED_DIR)

Project directory: c:\Users\pmpra\OneDrive\Desktop\Projects\PLG_Fearture_Engagement_Dashboard
Raw data directory: c:\Users\pmpra\OneDrive\Desktop\Projects\PLG_Fearture_Engagement_Dashboard\data\raw
Processed data directory: c:\Users\pmpra\OneDrive\Desktop\Projects\PLG_Fearture_Engagement_Dashboard\data\processed


In [2]:
users = pd.read_csv(RAW_DIR / "users.csv")
sessions = pd.read_csv(RAW_DIR / "sessions.csv")
feature_usage = pd.read_csv(RAW_DIR / "feature_usage.csv")
subscriptions = pd.read_csv(RAW_DIR / "subscriptions.csv")

print("All datasets loaded successfully!")

print("Users:", users.shape)
print("Sessions:", sessions.shape)
print("Feature usage:", feature_usage.shape)
print("Subscriptions:", subscriptions.shape)

All datasets loaded successfully!
Users: (5000, 6)
Sessions: (20000, 5)
Feature usage: (15000, 4)
Subscriptions: (5000, 5)


In [3]:
def standardize_columns(df):
    df = df.copy()

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )

    return df


users = standardize_columns(users)
sessions = standardize_columns(sessions)
feature_usage = standardize_columns(feature_usage)
subscriptions = standardize_columns(subscriptions)

print(users.columns.tolist())
print(sessions.columns.tolist())
print(feature_usage.columns.tolist())
print(subscriptions.columns.tolist())

['user_id', 'signup_date', 'plan_type', 'country', 'industry', 'acquisition_channel']
['session_id', 'user_id', 'date', 'duration_minutes', 'device_type']
['user_id', 'feature_name', 'usage_count', 'date']
['user_id', 'plan_start_date', 'plan_end_date', 'churned', 'mrr']


In [4]:
# User categories
users["plan_type"] = (
    users["plan_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

users["plan_type"] = users["plan_type"].replace(
    {"": pd.NA}
)

users["acquisition_channel"] = (
    users["acquisition_channel"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({"": pd.NA})
    .fillna("unknown")
)

users["country"] = (
    users["country"]
    .astype("string")
    .str.strip()
    .str.upper()
)

users["industry"] = (
    users["industry"]
    .astype("string")
    .str.strip()
)

# Session categories
sessions["device_type"] = (
    sessions["device_type"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({"": pd.NA})
    .fillna("unknown")
)

# Feature categories
feature_usage["feature_name"] = (
    feature_usage["feature_name"]
    .astype("string")
    .str.strip()
    .str.lower()
)

print(users["plan_type"].value_counts(dropna=False))
print(sessions["device_type"].value_counts(dropna=False))
print(feature_usage["feature_name"].value_counts(dropna=False))

plan_type
paid    2458
free    2441
<NA>     101
Name: count, dtype: Int64
device_type
mobile     6461
tablet     6434
desktop    6279
unknown     826
Name: count, dtype: Int64
feature_name
analytics      3139
reports        3038
export         2989
ai_insights    2923
dashboard      2911
Name: count, dtype: Int64


In [6]:
date_columns = {
    "users": ["signup_date"],
    "sessions": ["date"],
    "feature_usage": ["date"],
    "subscriptions": ["plan_start_date", "plan_end_date"],
}

dataframes = {
    "users": users,
    "sessions": sessions,
    "feature_usage": feature_usage,
    "subscriptions": subscriptions,
}

for table_name, columns in date_columns.items():
    df = dataframes[table_name]

    for col in columns:
        df[col] = pd.to_datetime(
            df[col],
            format="mixed",
            errors="coerce"
        )

        print(
            table_name,
            col,
            "| missing or invalid:",
            df[col].isna().sum()
        )

users signup_date | missing or invalid: 299
sessions date | missing or invalid: 629
feature_usage date | missing or invalid: 477
subscriptions plan_start_date | missing or invalid: 278
subscriptions plan_end_date | missing or invalid: 1818


In [7]:
# Sessions
sessions["duration_minutes"] = pd.to_numeric(
    sessions["duration_minutes"],
    errors="coerce"
)

# Feature usage
feature_usage["usage_count"] = pd.to_numeric(
    feature_usage["usage_count"],
    errors="coerce"
)

# Subscriptions
subscriptions["MRR"] = pd.to_numeric(
    subscriptions["MRR"],
    errors="coerce"
)

# Inspect invalid negative values
print(
    "Negative durations:",
    (sessions["duration_minutes"] < 0).sum()
)

print(
    "Negative usage counts:",
    (feature_usage["usage_count"] < 0).sum()
)

print(
    "Negative MRR values:",
    (subscriptions["MRR"] < 0).sum()
)

KeyError: 'MRR'

In [8]:
print(subscriptions.columns.tolist())

['user_id', 'plan_start_date', 'plan_end_date', 'churned', 'mrr']


In [9]:
subscriptions.columns = (
    subscriptions.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

print(subscriptions.columns.tolist())

['user_id', 'plan_start_date', 'plan_end_date', 'churned', 'mrr']


In [10]:
subscriptions["mrr"] = pd.to_numeric(
    subscriptions["mrr"],
    errors="coerce"
)

print("Missing MRR values:", subscriptions["mrr"].isna().sum())
print(subscriptions["mrr"].describe())

Missing MRR values: 143
count    4857.000000
mean       36.076590
std        45.103879
min         0.000000
25%         0.000000
50%         0.000000
75%        59.000000
max       149.000000
Name: mrr, dtype: float64


In [11]:
subscriptions["plan_start_date"]
subscriptions["plan_end_date"]
subscriptions["churned"]
subscriptions["mrr"]

0        0.0
1        0.0
2       59.0
3       99.0
4        0.0
        ... 
4995     0.0
4996    39.0
4997     0.0
4998     0.0
4999    39.0
Name: mrr, Length: 5000, dtype: float64

In [13]:
sessions.loc[
    sessions["duration_minutes"] < 0,
    "duration_minutes"
] = np.nan

feature_usage.loc[
    feature_usage["usage_count"] < 0,
    "usage_count"
] = np.nan

subscriptions.loc[
    subscriptions["mrr"] < 0,
    "mrr"
] = np.nan

In [14]:
subscriptions["churned"] = (
    subscriptions["churned"]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace({
        "Y": "yes",
        "N": "no",
        "YES": "yes",
        "NO": "no",
        "": pd.NA,
    })
)

print(subscriptions["churned"].value_counts(dropna=False))

churned
no      2436
yes     2429
<NA>     135
Name: count, dtype: Int64


In [15]:
print("Duplicate user IDs:", users["user_id"].duplicated().sum())
print("Duplicate session IDs:", sessions["session_id"].duplicated().sum())

print(
    "Unknown session user IDs:",
    (~sessions["user_id"].isin(users["user_id"])).sum()
)

print(
    "Unknown feature usage user IDs:",
    (~feature_usage["user_id"].isin(users["user_id"])).sum()
)

print(
    "Unknown subscription user IDs:",
    (~subscriptions["user_id"].isin(users["user_id"])).sum()
)

Duplicate user IDs: 0
Duplicate session IDs: 0
Unknown session user IDs: 0
Unknown feature usage user IDs: 0
Unknown subscription user IDs: 0


In [16]:
cleaned_data = {
    "users_clean.csv": users,
    "sessions_clean.csv": sessions,
    "feature_usage_clean.csv": feature_usage,
    "subscriptions_clean.csv": subscriptions,
}

for filename, df in cleaned_data.items():
    output_path = PROCESSED_DIR / filename
    df.to_csv(output_path, index=False)

    print(f"Saved: {output_path}")
    print("Rows:", len(df))

Saved: c:\Users\pmpra\OneDrive\Desktop\Projects\PLG_Fearture_Engagement_Dashboard\data\processed\users_clean.csv
Rows: 5000
Saved: c:\Users\pmpra\OneDrive\Desktop\Projects\PLG_Fearture_Engagement_Dashboard\data\processed\sessions_clean.csv
Rows: 20000
Saved: c:\Users\pmpra\OneDrive\Desktop\Projects\PLG_Fearture_Engagement_Dashboard\data\processed\feature_usage_clean.csv
Rows: 15000
Saved: c:\Users\pmpra\OneDrive\Desktop\Projects\PLG_Fearture_Engagement_Dashboard\data\processed\subscriptions_clean.csv
Rows: 5000


In [17]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd().parent
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

files = [
    "users_clean.csv",
    "sessions_clean.csv",
    "feature_usage_clean.csv",
    "subscriptions_clean.csv"
]

for filename in files:
    path = PROCESSED_DIR / filename

    if path.exists():
        df = pd.read_csv(path)
        print(f"\n{filename}")
        print("Rows and columns:", df.shape)
        print("Columns:", df.columns.tolist())
    else:
        print(f"\nMISSING FILE: {filename}")


users_clean.csv
Rows and columns: (5000, 6)
Columns: ['user_id', 'signup_date', 'plan_type', 'country', 'industry', 'acquisition_channel']

sessions_clean.csv
Rows and columns: (20000, 5)
Columns: ['session_id', 'user_id', 'date', 'duration_minutes', 'device_type']

feature_usage_clean.csv
Rows and columns: (15000, 4)
Columns: ['user_id', 'feature_name', 'usage_count', 'date']

subscriptions_clean.csv
Rows and columns: (5000, 5)
Columns: ['user_id', 'plan_start_date', 'plan_end_date', 'churned', 'mrr']
